In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import keras

from contextlib import ExitStack

# Keras imports used by the saved models
from keras.models import Sequential, Model
from keras.layers import (
    Dense,
    Dropout,
    Input,
    BatchNormalization,
    ReLU,
    LeakyReLU,
    Softmax,
    Activation,
)
from keras.callbacks import Callback, EarlyStopping
from keras.optimizers import Adam

ROOT = Path(".")

# Main + Purified models
pkl_paths = sorted(ROOT.glob("*.pkl")) + sorted((ROOT / "Purified").glob("*.pkl"))

# Helps some legacy Keras deserialization paths
try:
    keras.config.enable_unsafe_deserialization()
except Exception:
    pass


# =========================================================
# Custom objects needed for SciKeras/Keras joblib loading
# =========================================================
class HistoryLogger(Callback):
    def on_train_end(self, logs=None):
        try:
            self.model.final_history = self.model.history.history
        except Exception:
            pass


def create_nn(
    meta=None,
    neurons_layer1=256,
    neurons_layer2=128,
    dropout_rate1=0.3,
    dropout_rate2=0.3,
    learning_rate=0.002,
    activation="relu",
):
    n_features_in = None
    n_classes = None

    if isinstance(meta, dict):
        n_features_in = meta.get("n_features_in_")
        n_classes = meta.get("n_classes_")

    if n_features_in is None:
        n_features_in = 1
    if n_classes is None:
        n_classes = 2

    tf.keras.backend.clear_session()

    model = Sequential([
        Input(shape=(n_features_in,)),
        Dense(neurons_layer1, activation=activation),
        BatchNormalization(),
        Dropout(dropout_rate1),
        Dense(neurons_layer2, activation=activation),
        BatchNormalization(),
        Dropout(dropout_rate2),
        Dense(n_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def create_classifier(
    meta=None,
    hidden_dim=200,
    dropout_rate1=0.2,
    dropout_rate2=0.3,
    learning_rate=2e-5,
    activation="relu",
):
    n_features_in = None
    n_classes = None

    if isinstance(meta, dict):
        n_features_in = meta.get("n_features_in_")
        n_classes = meta.get("n_classes_")

    if n_features_in is None:
        n_features_in = 500
    if n_classes is None:
        n_classes = 2

    inp = Input(shape=(n_features_in,))
    x = BatchNormalization()(inp)
    x = Dropout(dropout_rate1)(x)

    if activation == "leakyrelu":
        x = Dense(hidden_dim)(x)
        x = LeakyReLU(negative_slope=0.01)(x)
    elif activation == "relu":
        x = Dense(hidden_dim)(x)
        x = ReLU()(x)
    else:
        x = Dense(hidden_dim, activation="tanh")(x)

    x = BatchNormalization()(x)
    x = Dropout(dropout_rate2)(x)
    out = Dense(n_classes, activation="softmax")(x)

    model = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


CUSTOM_OBJECTS = {
    "HistoryLogger": HistoryLogger,
    "create_nn": create_nn,
    "create_classifier": create_classifier,
    "Sequential": Sequential,
    "Model": Model,
    "Dense": Dense,
    "Dropout": Dropout,
    "Input": Input,
    "BatchNormalization": BatchNormalization,
    "ReLU": ReLU,
    "LeakyReLU": LeakyReLU,
    "Softmax": Softmax,
    "Activation": Activation,
    "Adam": Adam,
    "EarlyStopping": EarlyStopping,
}


def load_nn_pickle(path):
    with ExitStack() as stack:
        try:
            stack.enter_context(keras.saving.custom_object_scope(CUSTOM_OBJECTS))
        except Exception:
            pass

        try:
            stack.enter_context(keras.utils.custom_object_scope(CUSTOM_OBJECTS))
        except Exception:
            pass

        try:
            stack.enter_context(tf.keras.utils.custom_object_scope(CUSTOM_OBJECTS))
        except Exception:
            pass

        return joblib.load(path)


# =========================================================
# Generic helpers
# =========================================================
def to_serializable(obj):
    if obj is None:
        return None

    if isinstance(obj, (str, int, float, bool)):
        return obj

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, (np.bool_,)):
        return bool(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if hasattr(obj, "numpy"):
        try:
            return obj.numpy().tolist()
        except Exception:
            return str(obj)

    if isinstance(obj, tuple):
        return [to_serializable(x) for x in obj]

    if isinstance(obj, list):
        return [to_serializable(x) for x in obj]

    if isinstance(obj, dict):
        return {str(k): to_serializable(v) for k, v in obj.items()}

    if hasattr(obj, "get_params"):
        return repr(obj)

    return str(obj)


def safe_attr(obj, attr, default=None):
    try:
        return getattr(obj, attr, default)
    except Exception:
        return default


def parse_model_info(rel_path):
    stem = rel_path.stem
    top_folder = rel_path.parts[0] if len(rel_path.parts) > 1 else "."

    purified = "Purified" in rel_path.parts
    cohort = "Norway" if "Norway" in stem else "TCGA"
    sampler = "IRUS" if "_IRUS_" in stem else ("SMOTE" if "_SMOTE_" in stem else "None")
    uses_ae = "_AE_" in stem

    return {
        "group_folder": top_folder,
        "cohort": cohort,
        "purified": purified,
        "sampler": sampler,
        "uses_ae": uses_ae,
    }


def find_sampler_step(model):
    if hasattr(model, "named_steps"):
        if "irus" in model.named_steps:
            return "irus", model.named_steps["irus"]
        if "smote" in model.named_steps:
            return "smote", model.named_steps["smote"]
    return None, None


def find_nn_wrapper(model):
    if hasattr(model, "named_steps"):
        if "nn" in model.named_steps:
            return model.named_steps["nn"]
        for _, step in model.named_steps.items():
            if "kerasclassifier" in type(step).__name__.lower():
                return step
    return model


def get_learning_rate(optimizer):
    if optimizer is None:
        return None

    lr = safe_attr(optimizer, "learning_rate", None)
    if lr is None:
        return None

    try:
        if hasattr(lr, "numpy"):
            return float(lr.numpy())
        return float(lr)
    except Exception:
        return str(lr)


def get_layer_details_from_model(keras_model):
    layers = safe_attr(keras_model, "layers", None)
    if not layers:
        return []

    details = []
    for layer in layers:
        activation = safe_attr(layer, "activation", None)
        details.append({
            "name": safe_attr(layer, "name", None),
            "class_name": layer.__class__.__name__,
            "units": safe_attr(layer, "units", None),
            "activation": getattr(activation, "__name__", str(activation)),
            "rate": safe_attr(layer, "rate", None),
            "input_shape": str(safe_attr(layer, "input_shape", None)),
            "output_shape": str(safe_attr(layer, "output_shape", None)),
            "count_params": layer.count_params() if hasattr(layer, "count_params") else None,
        })
    return details


# =========================================================
# Fallback parser for failed Keras/SciKeras pickles
# This is what fixes the Purified non-AE failures.
# =========================================================
def parse_pickle_int(blob, key):
    key_bytes = key.encode("utf-8") + b"\x94"
    idx = blob.find(key_bytes)
    if idx == -1:
        return None

    j = idx + len(key_bytes)
    if j >= len(blob):
        return None

    op = blob[j:j+1]

    try:
        if op == b"K":  # 1-byte unsigned int
            return blob[j + 1]
        if op == b"M":  # 2-byte unsigned int
            return int.from_bytes(blob[j + 1:j + 3], "little", signed=False)
        if op == b"J":  # 4-byte signed int
            return int.from_bytes(blob[j + 1:j + 5], "little", signed=True)
    except Exception:
        return None

    return None


def extract_json_after_marker(text, marker):
    idx = text.find(marker)
    if idx == -1:
        return None

    start = text.find("{", idx)
    if start == -1:
        return None

    decoder = json.JSONDecoder()
    try:
        obj, _ = decoder.raw_decode(text[start:])
        return obj
    except Exception:
        return None


def normalize_activation(value):
    if isinstance(value, dict):
        class_name = value.get("class_name")
        if class_name:
            return class_name
        return str(value)
    return value


def compute_params_from_config_layers(layers):
    current_dim = None
    total_params = 0
    output_units = None
    layer_details = []
    input_shape = None
    output_shape = None

    for layer in layers:
        class_name = layer.get("class_name")
        conf = layer.get("config", {})

        units = conf.get("units")
        activation = normalize_activation(conf.get("activation"))
        rate = conf.get("rate")
        name = conf.get("name")

        count_params = None
        layer_output_shape = None

        if class_name == "InputLayer":
            batch_shape = conf.get("batch_shape")
            input_shape = batch_shape
            if isinstance(batch_shape, list) and len(batch_shape) >= 2:
                current_dim = batch_shape[-1]
                layer_output_shape = tuple(batch_shape)

        elif class_name == "Dense":
            if current_dim is not None and units is not None:
                use_bias = conf.get("use_bias", True)
                count_params = current_dim * units + (units if use_bias else 0)
                total_params += count_params
                current_dim = units
                output_units = units
                layer_output_shape = (None, units)

        elif class_name == "BatchNormalization":
            if current_dim is not None:
                count_params = 4 * current_dim
                total_params += count_params
                layer_output_shape = (None, current_dim)

        elif class_name in {"Dropout", "ReLU", "LeakyReLU", "Activation", "Softmax"}:
            if current_dim is not None:
                layer_output_shape = (None, current_dim)

        layer_details.append({
            "name": name,
            "class_name": class_name,
            "units": units,
            "activation": activation,
            "rate": rate,
            "input_shape": None,
            "output_shape": str(layer_output_shape) if layer_output_shape is not None else None,
            "count_params": count_params if count_params is not None else 0,
        })

        if layer_output_shape is not None:
            output_shape = layer_output_shape

    return {
        "input_shape": tuple(input_shape) if isinstance(input_shape, list) else input_shape,
        "output_shape": output_shape,
        "total_params": total_params if total_params != 0 else None,
        "n_features_in": input_shape[-1] if isinstance(input_shape, list) and len(input_shape) >= 2 else None,
        "n_classes": output_units,
        "layer_details": layer_details,
    }


def extract_fallback_from_pickle(full_path):
    blob = full_path.read_bytes()
    text = blob.decode("latin1", errors="ignore")

    metadata_json = extract_json_after_marker(text, "metadata.json")
    config_json = extract_json_after_marker(text, "config.json")

    batch_size = parse_pickle_int(blob, "batch_size")
    epochs = parse_pickle_int(blob, "epochs")
    random_state = parse_pickle_int(blob, "random_state")
    k_neighbors = parse_pickle_int(blob, "k_neighbors")

    result = {
        "load_method": "fallback_pickle_parse",
        "keras_version": metadata_json.get("keras_version") if isinstance(metadata_json, dict) else None,
        "batch_size": batch_size,
        "epochs": epochs,
        "random_state": random_state,
        "k_neighbors": k_neighbors,
        "keras_model_class": None,
        "input_shape": None,
        "output_shape": None,
        "n_features_in_": None,
        "n_classes": None,
        "optimizer": None,
        "learning_rate": None,
        "loss": None,
        "metrics": None,
        "num_layers": None,
        "total_trainable_params": None,
        "layer_details": [],
        "history_keys": None,
        "history_epochs_recorded": None,
    }

    if isinstance(config_json, dict):
        result["keras_model_class"] = config_json.get("class_name")

        model_cfg = config_json.get("config", {})
        layers = model_cfg.get("layers", [])
        layer_info = compute_params_from_config_layers(layers)

        result["input_shape"] = str(layer_info["input_shape"])
        result["output_shape"] = str(layer_info["output_shape"])
        result["n_features_in_"] = layer_info["n_features_in"]
        result["n_classes"] = layer_info["n_classes"]
        result["num_layers"] = len(layer_info["layer_details"])
        result["total_trainable_params"] = layer_info["total_params"]
        result["layer_details"] = layer_info["layer_details"]

        compile_cfg = config_json.get("compile_config", {})
        optimizer_cfg = compile_cfg.get("optimizer", {})

        if isinstance(optimizer_cfg, dict):
            result["optimizer"] = optimizer_cfg.get("class_name")
            opt_conf = optimizer_cfg.get("config", {})
            result["learning_rate"] = opt_conf.get("learning_rate")

        result["loss"] = compile_cfg.get("loss")
        result["metrics"] = str(compile_cfg.get("metrics"))

    return result


# =========================================================
# Inspect models
# =========================================================
summary_rows = []
all_params = {}
failed_models = []

for full_path in pkl_paths:
    rel_path = full_path.relative_to(ROOT)
    model_name = full_path.stem
    info = parse_model_info(rel_path)

    file_size_bytes = full_path.stat().st_size
    file_size_mb = file_size_bytes / (1024 ** 2)

    try:
        model = load_nn_pickle(full_path)

        has_named_steps = hasattr(model, "named_steps")
        step_names = list(model.named_steps.keys()) if has_named_steps else []

        sampler_name, sampler_step = find_sampler_step(model)
        nn_wrapper = find_nn_wrapper(model)
        keras_model = safe_attr(nn_wrapper, "model_", None)

        classes = safe_attr(nn_wrapper, "classes_", None)
        n_features_in = safe_attr(nn_wrapper, "n_features_in_", None)
        target_type = safe_attr(nn_wrapper, "target_type_", None)
        history = safe_attr(keras_model, "final_history", None) if keras_model is not None else None

        layer_details = get_layer_details_from_model(keras_model) if keras_model is not None else []
        total_params = keras_model.count_params() if keras_model is not None and hasattr(keras_model, "count_params") else None
        optimizer = safe_attr(keras_model, "optimizer", None) if keras_model is not None else None

        summary_rows.append({
            "model_name": model_name,
            "relative_path": str(rel_path),
            "group_folder": info["group_folder"],
            "cohort": info["cohort"],
            "purified": info["purified"],
            "sampler": info["sampler"],
            "uses_ae": info["uses_ae"],
            "load_method": "joblib_load",

            "file_size_bytes": int(file_size_bytes),
            "file_size_mb": round(file_size_mb, 2),

            "object_type": type(model).__name__,
            "steps": " -> ".join(step_names) if step_names else None,

            "sampler_name": sampler_name,
            "sampler_type": type(sampler_step).__name__ if sampler_step is not None else None,
            "sampling_strategy": str(safe_attr(sampler_step, "sampling_strategy", None)) if sampler_step is not None else None,
            "k_neighbors": safe_attr(sampler_step, "k_neighbors", None) if sampler_step is not None else None,

            "nn_wrapper_type": type(nn_wrapper).__name__,
            "n_features_in_": n_features_in,
            "n_classes": len(classes) if classes is not None else None,
            "classes": str(classes.tolist() if hasattr(classes, "tolist") else classes),
            "target_type_": target_type,

            "epochs": safe_attr(nn_wrapper, "epochs", None),
            "batch_size": safe_attr(nn_wrapper, "batch_size", None),
            "verbose": safe_attr(nn_wrapper, "verbose", None),
            "random_state": safe_attr(nn_wrapper, "random_state", None),

            "keras_model_class": type(keras_model).__name__ if keras_model is not None else None,
            "input_shape": str(safe_attr(keras_model, "input_shape", None)) if keras_model is not None else None,
            "output_shape": str(safe_attr(keras_model, "output_shape", None)) if keras_model is not None else None,
            "num_layers": len(layer_details) if layer_details else None,
            "total_trainable_params": total_params,

            "optimizer": optimizer.__class__.__name__ if optimizer is not None else None,
            "learning_rate": get_learning_rate(optimizer),
            "loss": str(safe_attr(keras_model, "loss", None)) if keras_model is not None else None,
            "metrics": str(safe_attr(keras_model, "metrics_names", None)) if keras_model is not None else None,

            "history_keys": str(list(history.keys())) if isinstance(history, dict) else None,
            "history_epochs_recorded": len(history.get("loss", [])) if isinstance(history, dict) and "loss" in history else None,
        })

        all_params[model_name] = {
            "relative_path": str(rel_path),
            "group_folder": info["group_folder"],
            "cohort": info["cohort"],
            "purified": info["purified"],
            "sampler": info["sampler"],
            "uses_ae": info["uses_ae"],
            "load_method": "joblib_load",

            "file_size_bytes": int(file_size_bytes),
            "file_size_mb": round(file_size_mb, 2),

            "model_type": type(model).__name__,
            "pipeline_steps": step_names,

            "pipeline_params": to_serializable(model.get_params(deep=True)) if hasattr(model, "get_params") else None,
            "sampler_params": to_serializable(sampler_step.get_params(deep=True)) if sampler_step is not None and hasattr(sampler_step, "get_params") else None,
            "nn_wrapper_params": to_serializable(nn_wrapper.get_params(deep=True)) if hasattr(nn_wrapper, "get_params") else None,

            "trained_attributes": {
                "classes_": to_serializable(classes),
                "n_features_in_": to_serializable(n_features_in),
                "target_type_": to_serializable(target_type),
                "input_shape": to_serializable(safe_attr(keras_model, "input_shape", None) if keras_model is not None else None),
                "output_shape": to_serializable(safe_attr(keras_model, "output_shape", None) if keras_model is not None else None),
                "total_trainable_params": to_serializable(total_params),
                "optimizer": to_serializable(optimizer.__class__.__name__ if optimizer is not None else None),
                "learning_rate": to_serializable(get_learning_rate(optimizer)),
                "loss": to_serializable(safe_attr(keras_model, "loss", None) if keras_model is not None else None),
                "metrics_names": to_serializable(safe_attr(keras_model, "metrics_names", None) if keras_model is not None else None),
                "history_keys": to_serializable(list(history.keys()) if isinstance(history, dict) else None),
                "history_epochs_recorded": to_serializable(len(history.get("loss", [])) if isinstance(history, dict) and "loss" in history else None),
            },

            "layer_details": to_serializable(layer_details),
            "training_history": to_serializable(history),
        }

    except Exception as e:
        try:
            fallback = extract_fallback_from_pickle(full_path)

            summary_rows.append({
                "model_name": model_name,
                "relative_path": str(rel_path),
                "group_folder": info["group_folder"],
                "cohort": info["cohort"],
                "purified": info["purified"],
                "sampler": info["sampler"],
                "uses_ae": info["uses_ae"],
                "load_method": fallback["load_method"],

                "file_size_bytes": int(file_size_bytes),
                "file_size_mb": round(file_size_mb, 2),

                "object_type": "Keras/SciKeras pickle (fallback parsed)",
                "steps": "sampler -> nn" if info["sampler"] != "None" else "nn",

                "sampler_name": info["sampler"].lower() if info["sampler"] != "None" else None,
                "sampler_type": "SMOTE" if info["sampler"] == "SMOTE" else ("RandomUnderSampler" if info["sampler"] == "IRUS" else None),
                "sampling_strategy": None,
                "k_neighbors": fallback["k_neighbors"],

                "nn_wrapper_type": "KerasClassifier (fallback)",
                "n_features_in_": fallback["n_features_in_"],
                "n_classes": fallback["n_classes"],
                "classes": None,
                "target_type_": None,

                "epochs": fallback["epochs"],
                "batch_size": fallback["batch_size"],
                "verbose": None,
                "random_state": fallback["random_state"],

                "keras_model_class": fallback["keras_model_class"],
                "input_shape": fallback["input_shape"],
                "output_shape": fallback["output_shape"],
                "num_layers": fallback["num_layers"],
                "total_trainable_params": fallback["total_trainable_params"],

                "optimizer": fallback["optimizer"],
                "learning_rate": fallback["learning_rate"],
                "loss": str(fallback["loss"]),
                "metrics": fallback["metrics"],

                "history_keys": fallback["history_keys"],
                "history_epochs_recorded": fallback["history_epochs_recorded"],
            })

            all_params[model_name] = {
                "relative_path": str(rel_path),
                "group_folder": info["group_folder"],
                "cohort": info["cohort"],
                "purified": info["purified"],
                "sampler": info["sampler"],
                "uses_ae": info["uses_ae"],
                "load_method": fallback["load_method"],

                "file_size_bytes": int(file_size_bytes),
                "file_size_mb": round(file_size_mb, 2),

                "model_type": "Keras/SciKeras pickle (fallback parsed)",
                "pipeline_steps": ["nn"] if info["sampler"] == "None" else [info["sampler"].lower(), "nn"],

                "pipeline_params": None,
                "sampler_params": {"k_neighbors": fallback["k_neighbors"]} if fallback["k_neighbors"] is not None else None,
                "nn_wrapper_params": {
                    "epochs": fallback["epochs"],
                    "batch_size": fallback["batch_size"],
                    "random_state": fallback["random_state"],
                },

                "trained_attributes": {
                    "n_features_in_": fallback["n_features_in_"],
                    "input_shape": fallback["input_shape"],
                    "output_shape": fallback["output_shape"],
                    "n_classes": fallback["n_classes"],
                    "total_trainable_params": fallback["total_trainable_params"],
                    "optimizer": fallback["optimizer"],
                    "learning_rate": fallback["learning_rate"],
                    "loss": fallback["loss"],
                    "metrics": fallback["metrics"],
                    "keras_version": fallback["keras_version"],
                },

                "layer_details": fallback["layer_details"],
                "training_history": None,
                "load_error": {
                    "error_type": type(e).__name__,
                    "error": str(e),
                    "error_repr": repr(e),
                },
            }

        except Exception as fallback_error:
            failed_models.append({
                "model_name": model_name,
                "relative_path": str(rel_path),
                "error_type": type(e).__name__,
                "error": str(e),
                "error_repr": repr(e),
                "fallback_error_type": type(fallback_error).__name__,
                "fallback_error": str(fallback_error),
                "fallback_error_repr": repr(fallback_error),
            })
            continue

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["cohort", "purified", "sampler", "uses_ae", "relative_path"]
).reset_index(drop=True)

failed_df = pd.DataFrame(failed_models)

pd.set_option("display.max_columns", None)

print("Loaded or parsed models:", len(summary_df))
print("Hard failures:", len(failed_df))

display(summary_df)

if len(failed_df) > 0:
    print("\nHard failures:")
    display(failed_df)

summary_csv = ROOT / "nn_model_summary.csv"
params_json = ROOT / "nn_model_params.json"
failed_csv = ROOT / "nn_model_failed_loads.csv"

summary_df.to_csv(summary_csv, index=False)

with open(params_json, "w", encoding="utf-8") as f:
    json.dump(all_params, f, indent=2, ensure_ascii=False)

if len(failed_df) > 0:
    failed_df.to_csv(failed_csv, index=False)

print("\nSaved:")
print(summary_csv)
print(params_json)
if len(failed_df) > 0:
    print(failed_csv)

if len(all_params) > 0:
    first_key = next(iter(all_params))
    print("\nExample:")
    print("Model:", first_key)
    print(json.dumps(all_params[first_key], indent=2, ensure_ascii=False))


c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator OrdinalEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.7.2 when using version 1.8.0. This might lead to breaking 

Loaded or parsed models: 24
Hard failures: 0


,model_name,relative_path,group_folder,cohort,purified,sampler,uses_ae,load_method,file_size_bytes,file_size_mb,object_type,steps,sampler_name,sampler_type,sampling_strategy,k_neighbors,nn_wrapper_type,n_features_in_,n_classes,classes,target_type_,epochs,batch_size,verbose,random_state,keras_model_class,input_shape,output_shape,num_layers,total_trainable_params,optimizer,learning_rate,loss,metrics,history_keys,history_epochs_recorded
0,NN_Norway_IRUS_RS_bestmodel,NN_Norway_IRUS_RS_bestmodel.pkl,.,Norway,False,IRUS,False,joblib_load,85940817,81.96,Pipeline,irus -> nn,irus,RandomUnderSampler,auto,NaN,KerasClassifier,20789,5,"[0, 1, 2, 3, 4]",multiclass,100,64,0.0,0,Sequential,"(None, 20789)","(None, 5)",7,5357317,Adam,0.00100,sparse_categorical_crossentropy,"['loss', 'compile_metrics']",None,None
1,NN_Norway_IRUS_RS_AE_bestmodel,NN_Norway_IRUS_RS_AE_bestmodel.pkl,.,Norway,False,IRUS,True,joblib_load,1699773,1.62,Pipeline,irus -> nn,irus,RandomUnderSampler,auto,NaN,KerasClassifier,500,5,"[0, 1, 2, 3, 4]",multiclass,100,16,0.0,0,Functional,"(None, 500)","(None, 5)",8,104005,Adam,0.00200,sparse_categorical_crossentropy,"['loss', 'compile_metrics']",None,None
2,NN_Norway_RS_bestmodel,NN_Norway_RS_bestmodel.pkl,.,Norway,False,None,False,joblib_load,85757651,81.78,Pipeline,nn,None,None,None,NaN,KerasClassifier,20789,5,"[0, 1, 2, 3, 4]",multiclass,100,16,0.0,0,Sequential,"(None, 20789)","(None, 5)",7,5357317,Adam,0.00010,sparse_categorical_crossentropy,"['loss', 'compile_metrics']",None,None
3,NN_Norway_RS_AE_bestmodel,NN_Norway_RS_AE_bestmodel.pkl,.,Norway,False,None,True,joblib_load,1696303,1.62,KerasClassifier,None,None,None,None,NaN,KerasClassifier,500,5,"[0, 1, 2, 3, 4]",multiclass,100,16,0.0,0,Functional,"(None, 500)","(None, 5)",7,104005,Adam,0.00010,sparse_categorical_crossentropy,"['loss', 'compile_metrics']",None,None
4,NN_Norway_SMOTE_RS_bestmodel,NN_Norway_SMOTE_RS_bestmodel.pkl,.,Norway,False,SMOTE,False,joblib_load,87604252,83.55,Pipeline,smote -> nn,smote,SMOTE,auto,5.0,KerasClassifier,20789,5,"[0, 1, 2, 3, 4]",multiclass,100,16,0.0,0,Sequential,"(None, 20789)","(None, 5)",7,5357317,Adam,0.00001,sparse_categorical_crossentropy,"['loss', 'compile_metrics']",None,None
5,NN_Norway_SMOTE_RS_AE_bestmodel,NN_Norway_SMOTE_RS_AE_bestmodel.pkl,.,Norway,False,SMOTE,True,joblib_load,1717774,1.64,Pipeline,smote -> nn,smote,SMOTE,auto,7.0,KerasClassifier,500,5,"[0, 1, 2, 3, 4]",multiclass,100,16,0.0,0,Functional,"(None, 500)","(None, 5)",7,104005,Adam,0.00200,sparse_categorical_crossentropy,"['loss', 'compile_metrics']",None,None
6,NN_Norway_IRUS_RS_Purified_bestmodel,Purified\NN_Norway_IRUS_RS_Purified_bestmodel.pkl,Purified,Norway,True,IRUS,False,fallback_pickle_parse,87222332,83.18,Keras/SciKeras pickle (fallback parsed),sampler -> nn,irus,RandomUnderSampler,None,NaN,KerasClassifier (fallback),21101,5,None,None,1,16,NaN,0,Sequential,"(None, 21101)","(None, 5)",8,5437189,Adam,0.00010,sparse_categorical_crossentropy,['accuracy'],None,None
7,NN_Norway_IRUS_RS_AE_Purified_bestmodel,Purified\NN_Norway_IRUS_RS_AE_Purified_bestmod...,Purified,Norway,True,IRUS,True,joblib_load,1697239,1.62,Pipeline,irus -> nn,irus,RandomUnderSampler,auto,NaN,KerasClassifier,500,5,"[0, 1, 2, 3, 4]",multiclass,100,16,0.0,0,Functional,"(None, 500)","(None, 5)",7,104005,Adam,0.00010,sparse_categorical_crossentropy,"['loss', 'compile_metrics']",None,None
8,NN_Norway_RS_Purified_bestmodel,Purified\NN_Norway_RS_Purified_bestmodel.pkl,Purified,Norway,True,None,False,fallback_pickle_parse,87046935,83.01,Keras/SciKeras pickle (fallback parsed),nn,None,None,None,NaN,KerasClassifier (fallback),21101,5,None,None,1,32,NaN,0,Sequential,"(None, 21101)","(None, 5)",8,5437189,Adam,0.00100,sparse_categorical_crossentropy,['accuracy'],None,None
9,NN_Norway_RS_AE_Purified_bestmodel,Purified\NN_Norway_RS_AE_Purified_bestmodel.pkl,Purified,Norway,True,None,True,joblib_load,1699557,1.62,KerasClassifier,None,None,None,None,NaN,KerasClassifier,500,5,"[0, 1, 2, 3, 4]",multiclass,10


Saved:
nn_model_summary.csv
nn_model_params.json

Example:
Model: NN_IRUS_RS_AE_bestmodel
{
  "relative_path": "NN_IRUS_RS_AE_bestmodel.pkl",
  "group_folder": ".",
  "cohort": "TCGA",
  "purified": false,
  "sampler": "IRUS",
  "uses_ae": true,
  "load_method": "joblib_load",
  "file_size_bytes": 1700999,
  "file_size_mb": 1.62,
  "model_type": "Pipeline",
  "pipeline_steps": [
    "irus",
    "nn"
  ],
  "pipeline_params": {
    "memory": null,
    "steps": [
      [
        "irus",
        "RandomUnderSampler(random_state=0)"
      ],
      [
        "nn",
        "KerasClassifier(\n\tmodel=<function create_classifier at 0x000001CB4B045DA0>\n\tbuild_fn=None\n\twarm_start=False\n\trandom_state=0\n\toptimizer=rmsprop\n\tloss=sparse_categorical_crossentropy\n\tmetrics=None\n\tbatch_size=32\n\tvalidation_batch_size=None\n\tverbose=0\n\tcallbacks=[<keras.src.callbacks.early_stopping.EarlyStopping object at 0x000001CB4B024740>, <__main__.HistoryLogger object at 0x000001CB01551E80>]\n\tva